In [0]:

from pyspark.sql import functions as F

print("=" * 55)
print("PRUEBAS AUTOMATIZADAS - FLIGHT DELAY BIG DATA")
print("=" * 55)

resultados = []

# ==========================================
# PRUEBA 1: CALIDAD DE SILVER
# ==========================================

try:
    silver = spark.table(
        "workspace.silver_flights.flights_clean"
    )

    columnas = {c.lower(): c for c in silver.columns}

    print("\nCOLUMNAS SILVER:")
    print(silver.columns)

    if "delayed_15" not in columnas:
        raise ValueError("No existe la etiqueta delayed_15")

    # Buscar el campo original del retraso.
    opciones = ["dep_delay", "depdelay", "departure_delay"]
    campo_delay = next(
        (columnas[c] for c in opciones if c in columnas),
        None
    )

    if campo_delay is None:
        raise ValueError(
            "No se encontró una columna de retraso para "
            "comprobar delayed_15"
        )

    errores = silver.filter(
        F.col(campo_delay).isNull()
        | F.col("delayed_15").isNull()
        | (
            F.col("delayed_15").cast("int")
            != F.when(
                F.col(campo_delay).cast("double") >= 15, 1
            ).otherwise(0)
        )
    ).count()

    if errores == 0:
        print("\nPRUEBA 1 - SILVER: PASS")
        print("Todas las etiquetas cumplen la regla de 15 minutos.")
        resultados.append(True)
    else:
        print("\nPRUEBA 1 - SILVER: FAIL")
        print("Registros con problemas:", errores)
        resultados.append(False)

except Exception as e:
    print("\nPRUEBA 1 - SILVER: FAIL")
    print(str(e))
    resultados.append(False)


# ==========================================
# PRUEBA 2: INTEGRIDAD DE GOLD
# ==========================================

try:
    gold = spark.table(
        "workspace.gold_flights.flights_weather_features"
    )

    total_silver = silver.count()
    total_gold = gold.count()

    cobertura = total_gold / total_silver if total_silver else 0

    print("\nPRUEBA 2 - INTEGRIDAD GOLD")
    print("Vuelos Silver:", total_silver)
    print("Vuelos Gold con clima:", total_gold)
    print("Cobertura:", round(cobertura * 100, 4), "%")

    # La integración no debería duplicar vuelos.
    # Se permite una pequeña pérdida por falta de pronóstico.
    valido = (
        total_silver > 0
        and total_gold <= total_silver
        and cobertura >= 0.99
    )

    if valido:
        print("\nPRUEBA 2 - GOLD: PASS")
        print("La integración conserva al menos el 99% de vuelos.")
        resultados.append(True)
    else:
        print("\nPRUEBA 2 - GOLD: FAIL")
        print("La cobertura o el volumen requieren revisión.")
        resultados.append(False)

except Exception as e:
    print("\nPRUEBA 2 - GOLD: FAIL")
    print(str(e))
    resultados.append(False)


# ==========================================
# RESUMEN FINAL
# ==========================================

print("\n" + "=" * 55)
print("RESULTADOS FINALES")
print("=" * 55)

print("Pruebas ejecutadas:", len(resultados))
print("Pruebas aprobadas:", sum(resultados))
print("Pruebas fallidas:", len(resultados) - sum(resultados))

if all(resultados):
    print("\nRESULTADO GENERAL: PASS")
else:
    print("\nRESULTADO GENERAL: FAIL")


PRUEBAS AUTOMATIZADAS - FLIGHT DELAY BIG DATA

COLUMNAS SILVER:
['flight_date', 'origin', 'dest', 'airline', 'tail_number', 'crs_dep_time', 'dep_delay', 'cancelled', 'source_year', 'source_month', 'departure_hour', 'departure_minute', 'year', 'month', 'day_of_week', 'delayed_15']

PRUEBA 1 - SILVER: PASS
Todas las etiquetas cumplen la regla de 15 minutos.

PRUEBA 2 - INTEGRIDAD GOLD
Vuelos Silver: 8655359
Vuelos Gold con clima: 8653172
Cobertura: 99.9747 %

PRUEBA 2 - GOLD: PASS
La integración conserva al menos el 99% de vuelos.

RESULTADOS FINALES
Pruebas ejecutadas: 2
Pruebas aprobadas: 2
Pruebas fallidas: 0

RESULTADO GENERAL: PASS
